In [ ]:
# ── All imports up-front ──────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.svm import SVC, SVR
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score,
                             roc_curve, auc, mean_squared_error, r2_score)
from sklearn.multiclass import OneVsRestClassifier

plt.rcParams['figure.dpi'] = 110
print("All libraries loaded ✓")

# Support Vector Machine (SVM) — Interactive Classroom Demo

## What is SVM?
A **Support Vector Machine** finds the **optimal hyperplane** that best separates classes by maximising the **margin** — the gap between the nearest data points (support vectors) of each class.

---

## The Kernel Trick
Real-world data is rarely linearly separable. The **kernel trick** maps data into a higher-dimensional space where a linear separator *does* exist — without ever computing the transformation explicitly.

| Kernel | Formula (conceptual) | When to use |
|--------|----------------------|-------------|
| **Linear** | `K(x,z) = x·z` | Linearly separable data, high-dimensional text/genomics |
| **RBF (Gaussian)** | `K(x,z) = exp(-γ‖x−z‖²)` | Most general-purpose cases, non-linear boundaries |
| **Polynomial** | `K(x,z) = (γx·z + r)^d` | Image classification, polynomial feature interactions |
| **Sigmoid** | `K(x,z) = tanh(γx·z + r)` | Neural-network-like behaviour |

### Key Hyperparameters
- **C** — Regularisation. High C → narrow margin, less misclassification (risk overfitting). Low C → wide margin, tolerates misclassification (risk underfitting).
- **γ (gamma)** — RBF/Poly/Sigmoid influence radius. High γ → each point has small reach (complex boundary). Low γ → smoother boundary.

---
**Dataset 1:** Red Wine Quality → SVC classification + GridSearchCV + ROC-AUC  
**Dataset 2:** Graduate Admission → SVR regression

In [ ]:
# Load the Red Wine Quality dataset from GitHub
data = pd.read_csv("https://raw.githubusercontent.com/aniruddhachoudhury/Red-Wine-Quality/master/winequality-red.csv")

print(f"Shape: {data.shape}")
print(f"\nQuality score distribution:\n{data['quality'].value_counts().sort_index()}")
data.head()

---
# PART 1 — Exploratory Data Analysis (Red Wine Quality Dataset)

The dataset contains physicochemical properties of red wine.  
**Target:** `quality` score (3–8). We will:
1. Explore the distribution
2. Convert to a **binary** target (good wine = quality ≥ 7) for clean ROC-AUC demonstration
3. Also keep the multi-class version for GridSearchCV accuracy tuning

In [ ]:
# Statistical summary — check ranges, means, and spread of features
data.describe().T

In [ ]:
# Visualise the class distribution and feature correlations
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: quality score counts
data['quality'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Wine Quality Score Distribution', fontsize=13)
axes[0].set_xlabel('Quality Score')
axes[0].set_ylabel('Count')
axes[0].axvline(x=3.5, color='red', linestyle='--', alpha=0.6, label='Good threshold (≥7)')
for p in axes[0].patches:
    axes[0].annotate(str(int(p.get_height())),
                     (p.get_x() + p.get_width()/2, p.get_height() + 5),
                     ha='center', fontsize=9)

# Right: correlation heatmap
corr = data.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1],
            linewidths=0.5, annot_kws={'size': 7})
axes[1].set_title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

# Key insight
top_corr = corr['quality'].abs().sort_values(ascending=False).drop('quality')
print("Top features correlated with quality:")
print(top_corr.head(5))

In [ ]:
# ── Feature matrix and target vectors ────────────────────────────────────────
X = data.drop("quality", axis=1)          # 11 physicochemical features
y_multi  = data["quality"]                # multi-class target  (3–8)
y_binary = (data["quality"] >= 7).astype(int)  # binary: 1=good, 0=average/poor

print("Feature columns:", list(X.columns))
print(f"\nMulti-class target  — unique values: {sorted(y_multi.unique())}")
print(f"Binary target       — class counts:\n{y_binary.value_counts().rename({0:'average/poor (0)', 1:'good (1)'})}")

---
# PART 2 — Data Preprocessing

**Steps:**
1. Separate features `X` from target `y`
2. Create a **binary target** (`good_wine`: 1 if quality ≥ 7, else 0) — needed for ROC-AUC
3. Train/Test split (67% / 33%)
4. **StandardScaler** — SVM is sensitive to feature scale; always scale before fitting!

In [ ]:
# ── Train / Test split ────────────────────────────────────────────────────────
# Same split for both multi-class and binary targets (same indices)
X_train, X_test, y_train_m, y_test_m = train_test_split(
    X, y_multi,  test_size=0.33, random_state=42)

_, _, y_train_b, y_test_b = train_test_split(
    X, y_binary, test_size=0.33, random_state=42)  # same random_state → same indices

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing  samples : {X_test.shape[0]}")

In [ ]:
# ── Feature Scaling (StandardScaler) ─────────────────────────────────────────
# Why scale? SVM uses distances (dot products). A feature with range [0,300]
# dominates one with range [0,1] without scaling.
# Rule: fit on TRAIN only, transform both train and test.

scaler = StandardScaler()
scaler.fit(X_train)              # learn mean & std from training data only

X_train_sc = scaler.transform(X_train)   # scale training set
X_test_sc  = scaler.transform(X_test)    # apply SAME scaling to test set

print("Means learned from training data:")
for feat, mean_ in zip(X.columns, scaler.mean_):
    print(f"  {feat:<25} {mean_:.4f}")
print("\nScaling done ✓")

In [ ]:
# ── Compare the 4 SVM kernels side by side ────────────────────────────────────
kernels = ['linear', 'rbf', 'poly', 'sigmoid']
results = []

for kernel in kernels:
    # Create SVC with default C=1 for fair comparison
    clf = SVC(kernel=kernel, random_state=42)
    clf.fit(X_train_sc, y_train_m)

    train_acc = clf.score(X_train_sc, y_train_m)
    test_acc  = clf.score(X_test_sc,  y_test_m)
    # 5-fold CV on the full scaled dataset for a robust estimate
    cv_scores = cross_val_score(SVC(kernel=kernel, random_state=42),
                                 X_train_sc, y_train_m, cv=5, scoring='accuracy')
    results.append({
        'Kernel'   : kernel,
        'Train Acc': round(train_acc, 4),
        'Test Acc' : round(test_acc, 4),
        'CV Mean'  : round(cv_scores.mean(), 4),
        'CV Std'   : round(cv_scores.std(), 4)
    })
    print(f"{kernel:<8}  train={train_acc:.3f}  test={test_acc:.3f}  "
          f"CV={cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

df_results = pd.DataFrame(results)
df_results

In [ ]:
# ── Visualise the kernel comparison ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))

x_pos = np.arange(len(kernels))
width = 0.28

bars1 = ax.bar(x_pos - width, df_results['Train Acc'], width, label='Train Acc', color='steelblue')
bars2 = ax.bar(x_pos,         df_results['Test Acc'],  width, label='Test Acc',  color='coral')
bars3 = ax.bar(x_pos + width, df_results['CV Mean'],   width, label='CV Mean',   color='seagreen',
               yerr=df_results['CV Std'], capsize=4)

ax.set_xticks(x_pos)
ax.set_xticklabels([k.capitalize() for k in kernels], fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('SVM Kernel Comparison (multi-class wine quality)', fontsize=13)
ax.set_ylim(0.4, 0.85)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.4)

# Annotate bars with values
for bar_group in [bars1, bars2, bars3]:
    for bar in bar_group:
        ax.annotate(f'{bar.get_height():.3f}',
                    xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', fontsize=7.5)
plt.tight_layout()
plt.show()

print("\n💡 Observation: RBF typically gives the best generalisation on this dataset."
      "\n   Large gap between Train and Test Acc → signs of overfitting (high C or complex kernel).")

---
## Interactive Kernel Demo — 2D Decision Boundaries

To *see* what kernels do, we reduce to 2 features (`alcohol` vs `volatile acidity` — top two correlated with quality) and plot decision regions for each kernel.

> In a real project you use all features; this is purely for visual intuition.

In [ ]:
# ── 2D Decision Boundary — one plot per kernel ───────────────────────────────
# Using only 2 features so we can plot on a 2D plane
feat_a, feat_b = 'alcohol', 'volatile acidity'
X2 = data[[feat_a, feat_b]].values
y2 = y_binary.values                       # binary for clean visualisation

sc2 = StandardScaler()
X2_sc = sc2.fit_transform(X2)

X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2_sc, y2, test_size=0.33, random_state=42)

kernels_2d = ['linear', 'rbf', 'poly', 'sigmoid']
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Mesh for background colouring
h = 0.04
x_min, x_max = X2_sc[:, 0].min() - 0.5, X2_sc[:, 0].max() + 0.5
y_min, y_max = X2_sc[:, 1].min() - 0.5, X2_sc[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

for ax, kernel in zip(axes, kernels_2d):
    clf2d = SVC(kernel=kernel, C=1.0, gamma='scale', random_state=42)
    clf2d.fit(X2_tr, y2_tr)

    Z = clf2d.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlGn')
    scatter = ax.scatter(X2_te[:, 0], X2_te[:, 1],
                         c=y2_te, cmap='RdYlGn', edgecolors='k', s=20, linewidths=0.5)
    acc = clf2d.score(X2_te, y2_te)
    ax.set_title(f'{kernel.capitalize()} kernel\nTest Acc = {acc:.3f}', fontsize=11)
    ax.set_xlabel(feat_a)
    ax.set_ylabel(feat_b)

plt.suptitle('Decision Boundaries — 2 Feature Projection (binary: good vs not)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Green = Good wine (quality ≥ 7)  |  Red = Average/Poor wine")

---
## Effect of C on the RBF Kernel — Bias-Variance Trade-off

- **Small C** → wide margin → allows misclassifications → high bias, low variance (underfitting)  
- **Large C** → narrow margin → fits training data tightly → low bias, high variance (overfitting)

In [ ]:
# ── Effect of C on RBF boundary ───────────────────────────────────────────────
C_values = [0.01, 0.1, 1, 10, 100]
fig, axes = plt.subplots(1, len(C_values), figsize=(20, 4))

for ax, C in zip(axes, C_values):
    clf_c = SVC(kernel='rbf', C=C, gamma='scale', random_state=42)
    clf_c.fit(X2_tr, y2_tr)

    Z = clf_c.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlGn')
    ax.scatter(X2_te[:, 0], X2_te[:, 1],
               c=y2_te, cmap='RdYlGn', edgecolors='k', s=18, linewidths=0.5)

    train_acc = clf_c.score(X2_tr, y2_tr)
    test_acc  = clf_c.score(X2_te, y2_te)
    ax.set_title(f'C = {C}\nTrain={train_acc:.3f}  Test={test_acc:.3f}', fontsize=10)
    ax.set_xlabel(feat_a)
    ax.set_ylabel(feat_b)

plt.suptitle('RBF Kernel — Effect of Regularisation Parameter C', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Notice: as C increases, the boundary becomes more jagged (overfits training data).")

---
# PART 3 — SVM Kernel Comparison

We train SVC with all four kernels on the **multi-class** problem and compare:
- Training accuracy
- Test accuracy
- Cross-validation score (more reliable than a single split)

---
# PART 4 — Hyperparameter Tuning with GridSearchCV

**GridSearchCV** exhaustively tries every combination from a parameter grid and picks the best via cross-validation.

```
param_grid = {
    'C'     : [0.1, 1, 10, 100],
    'gamma' : ['scale', 'auto', 0.01, 0.1],
    'kernel': ['rbf', 'linear', 'poly']
}
```

Each combination is evaluated with **5-fold cross-validation** → `4 × 4 × 3 × 5 = 240 model fits`.

In [ ]:
# ── GridSearchCV — hyperparameter tuning ──────────────────────────────────────
param_grid = {
    'C'     : [0.1, 1, 10, 100],
    'gamma' : ['scale', 'auto', 0.01, 0.1],
    'kernel': ['rbf', 'linear', 'poly']
}

print("Running GridSearchCV (240 fits) — this may take ~1–2 minutes …")
grid_search = GridSearchCV(
    estimator  = SVC(random_state=42),
    param_grid = param_grid,
    cv         = 5,             # 5-fold cross-validation
    scoring    = 'accuracy',
    n_jobs     = -1,            # use all CPU cores
    verbose    = 1
)
grid_search.fit(X_train_sc, y_train_m)

print(f"\n✅ Best parameters : {grid_search.best_params_}")
print(f"   Best CV accuracy: {grid_search.best_score_:.4f}")

In [ ]:
# ── Evaluate the best model from GridSearchCV ─────────────────────────────────
best_svc = grid_search.best_estimator_          # pre-fitted on full training set

y_pred_best = best_svc.predict(X_test_sc)
baseline_acc = accuracy_score(y_test_m, SVC(random_state=42).fit(X_train_sc, y_train_m).predict(X_test_sc))

print(f"Baseline SVC (default params) test accuracy : {baseline_acc:.4f}")
print(f"Tuned    SVC (GridSearchCV)   test accuracy : {accuracy_score(y_test_m, y_pred_best):.4f}")
print(f"\nClassification Report:\n")
print(classification_report(y_test_m, y_pred_best))

# Confusion matrix
cm = confusion_matrix(y_test_m, y_pred_best)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=sorted(y_test_m.unique()),
            yticklabels=sorted(y_test_m.unique()))
plt.title('Confusion Matrix — Tuned SVC (multi-class)')
plt.xlabel('Predicted Quality')
plt.ylabel('True Quality')
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualise GridSearch results: C vs accuracy per kernel ────────────────────
cv_results = pd.DataFrame(grid_search.cv_results_)

# Pivot: average CV score for each (kernel, C) combination (averaged over gamma)
pivot_data = cv_results.groupby(['param_kernel', 'param_C'])['mean_test_score'].mean().unstack()

fig, ax = plt.subplots(figsize=(8, 4))
for kernel_name in pivot_data.index:
    ax.plot(pivot_data.columns, pivot_data.loc[kernel_name],
            marker='o', label=kernel_name.capitalize())

ax.set_xscale('log')
ax.set_xlabel('C (regularisation)', fontsize=12)
ax.set_ylabel('Mean CV Accuracy', fontsize=12)
ax.set_title('GridSearchCV: Effect of C on CV Accuracy per Kernel', fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.4)
plt.tight_layout()
plt.show()

---
# PART 5 — ROC-AUC Curve

## What is ROC-AUC?
- **ROC** = Receiver Operating Characteristic curve  
- Plots **True Positive Rate (Recall)** vs **False Positive Rate** at every possible decision threshold  
- **AUC** = Area Under Curve → single scalar summary (0.5 = random, 1.0 = perfect)  

### Binary Classification (good wine vs not)
We use `SVC(probability=True)` so we get probability scores (not just hard labels), which are needed to draw the ROC curve.

### Multiclass — One-vs-Rest (OvR)
For 6 classes we compute one ROC curve per class (treating that class as positive vs all others).

In [ ]:
# ── ROC-AUC: Binary Classification (good wine = 1) ───────────────────────────
# probability=True enables Platt scaling so we get class probabilities
svc_bin = SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42)
svc_bin.fit(X_train_sc, y_train_b)

# Probability of positive class (good wine)
y_prob_bin = svc_bin.predict_proba(X_test_sc)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test_b, y_prob_bin)
roc_auc_bin = auc(fpr, tpr)

# Also compute for Logistic Regression as baseline comparison
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train_b)
y_prob_lr = lr.predict_proba(X_test_sc)[:, 1]
fpr_lr, tpr_lr, _ = roc_curve(y_test_b, y_prob_lr)
auc_lr = auc(fpr_lr, tpr_lr)

# Plot
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr,    tpr,    color='darkorange', lw=2, label=f'SVC RBF  (AUC = {roc_auc_bin:.3f})')
ax.plot(fpr_lr, tpr_lr, color='steelblue',  lw=2, label=f'Logistic Reg (AUC = {auc_lr:.3f})')
ax.plot([0,1], [0,1], 'k--', lw=1, label='Random classifier (AUC = 0.5)')

ax.fill_between(fpr, tpr, alpha=0.08, color='darkorange')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curve — Binary: Good Wine (quality ≥ 7)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"SVC  binary AUC : {roc_auc_bin:.4f}")
print(f"LR   binary AUC : {auc_lr:.4f}")

In [ ]:
# ── ROC-AUC: Multiclass One-vs-Rest ───────────────────────────────────────────
# Binarise labels: create a column per class
classes = sorted(y_train_m.unique())   # [3, 4, 5, 6, 7, 8]
Y_test_bin = label_binarize(y_test_m, classes=classes)   # shape (n_samples, n_classes)

# Train OvR SVC with probability
ovr_clf = OneVsRestClassifier(
    SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42)
)
ovr_clf.fit(X_train_sc, y_train_m)
y_score = ovr_clf.predict_proba(X_test_sc)       # shape (n_samples, n_classes)

# Compute ROC curve and AUC per class
colors = ['#e41a1c','#ff7f00','#4daf4a','#377eb8','#984ea3','#a65628']
fig, ax = plt.subplots(figsize=(8, 6))

for i, (cls, color) in enumerate(zip(classes, colors)):
    fpr_i, tpr_i, _ = roc_curve(Y_test_bin[:, i], y_score[:, i])
    auc_i = auc(fpr_i, tpr_i)
    ax.plot(fpr_i, tpr_i, color=color, lw=2, label=f'Quality {cls}  (AUC = {auc_i:.2f})')

ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Multiclass ROC Curves — One-vs-Rest SVC (RBF kernel)', fontsize=13)
ax.legend(fontsize=10, loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Macro-average AUC
macro_auc = roc_auc_score(Y_test_bin, y_score, average='macro')
print(f"Macro-average AUC (OvR): {macro_auc:.4f}")

---
# PART 6 — Support Vector Regression (SVR)

## SVC vs SVR
| | SVC (Classifier) | SVR (Regressor) |
|-|-----------------|-----------------|
| **Output** | Discrete class label | Continuous numeric value |
| **Loss** | Hinge loss (misclassification) | ε-insensitive loss (prediction tube) |
| **Goal** | Maximise margin between classes | Fit data within an ε-tube, penalise points outside |
| **Key param** | C, kernel, γ | C, ε (epsilon), kernel, γ |

### ε (epsilon) tube
Points that fall **inside** the tube around the prediction line contribute **zero loss**.  
Only points **outside** the tube are penalised — this gives SVR its sparsity.

**Dataset:** Graduate Admission Prediction  
**Target:** `Chance of Admit` (continuous, 0–1)

In [ ]:
# ── Load Graduate Admission dataset ──────────────────────────────────────────
admit = pd.read_csv("https://raw.githubusercontent.com/srinivasav22/Graduate-Admission-Prediction/master/Admission_Predict_Ver1.1.csv")

# 'Serial No.' is just a row index — drop it
admit.drop(columns=['Serial No.'], inplace=True)

# Strip any trailing spaces from column names
admit.columns = admit.columns.str.strip()

print(f"Shape: {admit.shape}")
print(f"\nMissing values:\n{admit.isnull().sum()}")
admit.head()

In [ ]:
# ── EDA — Admission dataset ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribution of the target
admit['Chance of Admit'].hist(bins=25, color='steelblue', edgecolor='white', ax=axes[0])
axes[0].set_title('Distribution of Chance of Admit', fontsize=13)
axes[0].set_xlabel('Chance of Admit')
axes[0].set_ylabel('Frequency')

# Correlation with target
corr_admit = admit.corr()['Chance of Admit'].drop('Chance of Admit').sort_values()
corr_admit.plot(kind='barh', color=['coral' if v < 0 else 'steelblue' for v in corr_admit],
                ax=axes[1], edgecolor='white')
axes[1].set_title('Feature Correlation with Chance of Admit', fontsize=13)
axes[1].set_xlabel('Pearson Correlation')
axes[1].axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

print("\nDescriptive stats:")
admit.describe().T

In [ ]:
# ── Prepare data for SVR ──────────────────────────────────────────────────────
X_adm = admit.drop('Chance of Admit', axis=1)
y_adm = admit['Chance of Admit']

X_adm_train, X_adm_test, y_adm_train, y_adm_test = train_test_split(
    X_adm, y_adm, test_size=0.2, random_state=42)

# Scale features — critical for SVR too
scaler_adm = StandardScaler()
X_adm_train_sc = scaler_adm.fit_transform(X_adm_train)
X_adm_test_sc  = scaler_adm.transform(X_adm_test)

print(f"Train: {X_adm_train_sc.shape}  |  Test: {X_adm_test_sc.shape}")

In [ ]:
# ── SVR — Compare kernels ─────────────────────────────────────────────────────
svr_kernels = ['linear', 'rbf', 'poly', 'sigmoid']
svr_results = []

for kernel in svr_kernels:
    svr = SVR(kernel=kernel, C=10, epsilon=0.01, gamma='scale')
    svr.fit(X_adm_train_sc, y_adm_train)

    y_pred_svr = svr.predict(X_adm_test_sc)
    rmse = np.sqrt(mean_squared_error(y_adm_test, y_pred_svr))
    r2   = r2_score(y_adm_test, y_pred_svr)
    svr_results.append({'Kernel': kernel, 'RMSE': round(rmse, 5), 'R²': round(r2, 4)})
    print(f"{kernel:<8}  RMSE={rmse:.5f}  R²={r2:.4f}")

df_svr = pd.DataFrame(svr_results)
print()
df_svr

In [ ]:
# ── SVR GridSearchCV ──────────────────────────────────────────────────────────
param_grid_svr = {
    'kernel' : ['linear', 'rbf'],
    'C'      : [1, 10, 100],
    'epsilon': [0.001, 0.01, 0.1],
    'gamma'  : ['scale', 'auto']
}

print("Running GridSearchCV for SVR …")
grid_svr = GridSearchCV(
    SVR(), param_grid_svr,
    cv=5, scoring='r2', n_jobs=-1, verbose=0
)
grid_svr.fit(X_adm_train_sc, y_adm_train)

print(f"\n✅ Best params : {grid_svr.best_params_}")
print(f"   Best CV R²  : {grid_svr.best_score_:.4f}")

In [ ]:
# ── Evaluate best SVR model ───────────────────────────────────────────────────
best_svr = grid_svr.best_estimator_
y_pred_best_svr = best_svr.predict(X_adm_test_sc)

rmse_best = np.sqrt(mean_squared_error(y_adm_test, y_pred_best_svr))
r2_best   = r2_score(y_adm_test, y_pred_best_svr)

print(f"Tuned SVR — RMSE: {rmse_best:.5f}  |  R²: {r2_best:.4f}")

# Actual vs Predicted scatter
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Actual vs Predicted
axes[0].scatter(y_adm_test, y_pred_best_svr, alpha=0.6, color='steelblue', edgecolors='white', s=40)
axes[0].plot([y_adm_test.min(), y_adm_test.max()],
             [y_adm_test.min(), y_adm_test.max()], 'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Chance of Admit', fontsize=12)
axes[0].set_ylabel('Predicted Chance of Admit', fontsize=12)
axes[0].set_title(f'SVR: Actual vs Predicted  (R²={r2_best:.4f})', fontsize=12)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: Residuals
residuals = y_adm_test.values - y_pred_best_svr
axes[1].scatter(y_pred_best_svr, residuals, alpha=0.6, color='darkorange', edgecolors='white', s=40)
axes[1].axhline(0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted', fontsize=12)
axes[1].set_ylabel('Residual (actual − predicted)', fontsize=12)
axes[1].set_title('Residual Plot — Tuned SVR', fontsize=12)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Sample predictions vs actuals
comparison = pd.DataFrame({
    'Actual'   : y_adm_test.values[:10],
    'Predicted': np.round(y_pred_best_svr[:10], 4)
})
print("\nSample Predictions vs Actuals:")
print(comparison.to_string(index=False))

---
# Summary & Key Takeaways

| Concept | What we demonstrated |
|---------|---------------------|
| **SVM margin** | The hyperplane that maximises the gap between support vectors |
| **Kernel trick** | Maps data to higher dimensions without explicit transformation |
| **Linear kernel** | Best for linearly separable / high-dim sparse data |
| **RBF kernel** | Best general-purpose kernel; controlled by C and γ |
| **Polynomial kernel** | Captures polynomial feature interactions |
| **C parameter** | Regularisation: low C → wide margin (underfitting), high C → narrow margin (overfitting) |
| **StandardScaler** | Always scale before SVM — distances are scale-sensitive |
| **GridSearchCV** | Systematic hyperparameter search with cross-validation |
| **ROC-AUC** | Threshold-independent performance measure; 1.0 = perfect |
| **SVR** | SVM extended to regression using ε-insensitive loss tube |

### Further Reading
- scikit-learn SVC: https://scikit-learn.org/stable/modules/svm.html  
- GridSearchCV: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html